# Behavioural rules of each cluster


The log skeleton discovered for each cluster separately, to read the rules that describe it: which
activities always follow or precede one another, which never occur together, and how often each
activity occurs. This is the qualitative counterpart of the conformance features of `06_clustering`.

* **Input:** `logs_w_cluster/<coding scheme>.csv`
* **Output:** `results/tables/log_skeleton_rules.csv`
* **Check:** the rules of a cluster agree with its label distribution in `07_stats_and_profiles`


In [1]:
from pathlib import Path
import os
import json
import warnings

import pandas as pd
import pm4py

warnings.filterwarnings('ignore')

# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

IN_FOLDER = 'logs_w_cluster'
CODING_SCHEMES = ['lin_dialogue_acts',
                  'acosta_epistemic_stance',
                  'del_bosque_cma']

PROFILE_NAMES = {
    'lin_dialogue_acts': {0: 'Extended-engagement', 1: 'Task-completion', 2: 'Front-loading'},
    'acosta_epistemic_stance': {0: 'Assignment-driven', 1: 'Gap-filling', 2: 'Sense-making'},
    'del_bosque_cma': {0: 'Tutored', 1: 'Passive', 2: 'Collaborative'},
}

FAMILIES = ['equivalence', 'always_after', 'always_before', 'never_together', 'directly_follows',
            'activ_freq']

In [2]:
# Relative path
path_conversations = DATA / IN_FOLDER
logs = {}
coding_schemes = {}
for code in CODING_SCHEMES:
    path_coding_scheme = Path('..') / "coding_schemes" / code

    # Read dataframe
    df = pd.read_csv(path_conversations / (code + '.csv'))

    logs[code] = df

    with open(path_coding_scheme / 'tags.json') as f:
        coding_scheme = json.load(f)
        coding_schemes[code] = coding_scheme

logs['acosta_epistemic_stance'].head()

,case_id,activity_id,ocel:eid,ocel:type:model,ocel:type:user,ocel:type:conversation,ocel:activity,ocel:timestamp,sentence_en,role,user_id,label,type,cluster
0,1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,1,['model_s108_1'],NaN,['1'],Other,2024-01-01 00:00:00,"Hello! 😸 I am SIMBA, and I will help you think...",model,s108,Other acts that maintain discussion,Other,0
1,1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,2,['model_s108_1'],NaN,['1'],K-,2024-01-01 00:01:00,What are the 2 to 5 main ideas to remember fro...,model,s108,Utterances that are hedges or requests,K-,0
2,1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,3,['model_s108_1'],NaN,['1'],Other,2024-01-01 00:02:00,I am here to accompany you in this reflection! 🌟,model,s108,Other acts that maintain discussion,Other,0
3,1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,4,NaN,['user_s108_1'],['1'],K-,2024-01-01 00:03:00,Can you clarify the difference between an isol...,user,s108,Utterances that are hedges or requests,K-,0
4,1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,5,['model_s108_1'],NaN,['1'],Reply,2024-01-01 00:04:00,Sure! 🌟,model,s108,Basic and simple replies to questions,Reply,0


## Discover one log skeleton per cluster

In [3]:
rows = []

for code in CODING_SCHEMES:
    df = logs[code]
    clusters = sorted(df['cluster'].unique())
    results = []

    for c in clusters:
        aux = df[df['cluster'] == c].copy()
        aux['ocel:timestamp'] = pd.to_datetime(aux['ocel:timestamp'])
        aux['case_id'] = aux['case_id'].astype(str)
        # Concatenate the activity with the role of the speaker
        aux['activity'] = aux['role'].str[:1] + '_' + aux['type']

        res = pm4py.discover_log_skeleton(
                aux,
                noise_threshold=0.4,  # THIS IS THE HYPERPARAMETER
                activity_key="activity",
                case_id_key="case_id",
                timestamp_key="ocel:timestamp",
            )

        results.append(res)

        for family in FAMILIES:
            if family == 'activ_freq':
                for activity, frequencies in res[family].items():
                    rows.append({'coding_scheme': code, 'cluster': c,
                                 'profile': PROFILE_NAMES[code][c], 'rule': family,
                                 'activity_a': activity, 'activity_b': '',
                                 'detail': sorted(frequencies)})
            else:
                for activity_a, activity_b in sorted(res[family]):
                    rows.append({'coding_scheme': code, 'cluster': c,
                                 'profile': PROFILE_NAMES[code][c], 'rule': family,
                                 'activity_a': activity_a, 'activity_b': activity_b, 'detail': ''})

rules = pd.DataFrame(rows)

path_tables = DATA / 'results' / 'tables'
path_tables.mkdir(parents=True, exist_ok=True)
rules.to_csv(path_tables / 'log_skeleton_rules.csv', index=False)

rules.groupby(['coding_scheme', 'profile', 'rule']).size().rename('n_rules').unstack(fill_value=0)

rule                                         activ_freq  always_after   
coding_scheme           profile                                         
acosta_epistemic_stance Assignment-driven            10            66  \
                        Gap-filling                  10            55   
                        Sense-making                 10            38   
del_bosque_cma          Collaborative                 9            30   
                        Passive                      12            73   
                        Tutored                       9            48   
lin_dialogue_acts       Extended-engagement          12            30   
                        Front-loading                22           101   
                        Task-completion              17            72   

rule                                         always_before  directly_follows   
coding_scheme           profile                                                
acosta_epistemic_stance Assignment-driven               59                 3  \
                        Gap-filling                     55                 6   
                        Sense-making                    35                 6   
del_bosque_cma          Collaborative                   34                 4   
                        Passive                         62                 5   
                        Tutored                         50                 3   
lin_dialogue_acts       Extended-engagement             37                 8   
                        Front-loading                  152                 8   
                        Task-completion                 95                 7   

rule                                         equivalence  never_together  
coding_scheme           profile                                           
acosta_epistemic_stance Assignment-driven              0              54  
                        Gap-filling                    0              67  
                        Sense-making                   4              70  
del_bosque_cma          Collaborative                  2              55  
                        Passive                        3              98  
                        Tutored                        0              51  
lin_dialogue_acts       Extended-engagement            9             109  
                        Front-loading                 12             317  
                        Task-completion                5             201

## How often each activity is allowed to occur, per cluster

In [4]:
activity_frequencies = rules[rules['rule'] == 'activ_freq'][
    ['coding_scheme', 'profile', 'activity_a', 'detail']].rename(
    columns={'activity_a': 'activity', 'detail': 'allowed_counts'})

activity_frequencies.head(30)

,coding_scheme,profile,activity,allowed_counts
193,lin_dialogue_acts,Extended-engagement,m_Statement,"[0, 1, 2, 3, 6]"
194,lin_dialogue_acts,Extended-engagement,m_Question,"[0, 1, 2, 3, 4]"
195,lin_dialogue_acts,Extended-engagement,u_Question,"[0, 3]"
196,lin_dialogue_acts,Extended-engagement,m_Acknowledge,"[0, 3]"
197,lin_dialogue_acts,Extended-engagement,m_Hint,"[0, 4]"
198,lin_dialogue_acts,Extended-engagement,u_Hint,"[0, 1, 2, 3]"
199,lin_dialogue_acts,Extended-engagement,m_Positive Feedback,"[0, 1, 2, 3]"
200,lin_dialogue_acts,Extended-engagement,m_Directive,"[0, 1]"
201,lin_dialogue_acts,Extended-engagement,m_Lukewarm Feedback,"[0, 1]"
202,lin_dialogue_acts,Extended-engagement,m_Negative Feedback,"[0, 1]"
